### Question
A marketing team sends promo emails. Find all users who received at least one promo email but never made a purchase within 7 days of receiving any of their emails. (A purchase counts only if it falls in the window [email_date, email_date + 7 days].)

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW sent_emails AS
SELECT *
FROM VALUES
    (1, DATE '2024-01-01'),
    (1, DATE '2024-02-01'),
    (2, DATE '2024-01-05'),
    (3, DATE '2024-01-10'),
    (4, DATE '2024-03-01'),
    (5, DATE '2024-01-01')

AS sent_emails(user_id, email_date);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW purchases AS
SELECT *
FROM VALUES
    (1, DATE '2024-01-03'),
    (2, DATE '2024-01-20'),
    (3, DATE '2024-02-15'),
    (5, DATE '2024-01-06'),
    (6, DATE '2024-01-06')
AS purchases(user_id, purchase_date);

### Edge case missing

In [0]:
%sql
select e.* from sent_emails e
where not exists (
select 1 from purchases p where e.user_id = p.user_id
and p.purchase_date between e.email_date AND e.email_date + interval 7 days
)

In [0]:
%sql
with valid_user as (select e.* from sent_emails e
join purchases p on e.user_id = p.user_id
and p.purchase_date between e.email_date AND e.email_date + interval 7 days
)
select 
e.* from  sent_emails e
where not exists 
(
select 1 from valid_user p 
where p.user_id = e.user_id
)


In [0]:
%sql
with valid_user as (select e.* from sent_emails e
join purchases p on e.user_id = p.user_id
and p.purchase_date between e.email_date AND e.email_date + interval 7 days
)
select 
e.* from  sent_emails e
left join valid_user v on e.user_id = v.user_id
where v.user_id is null


### Question: Users Who Never Logged in From a New Device
A security team wants to identify trusted users.

A user is considered untrusted if any of their login sessions occurred from a device that had never been used by that user before.

Find all users who have logged in at least once and never logged in from a new device.

A device is considered "new" only on its first login for that user. Any later login from the same device is not new.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW login_history AS
SELECT *
FROM VALUES
    (1, DATE '2024-01-01', 'Laptop'),
    (1, DATE '2024-01-05', 'Laptop'),
    (1, DATE '2024-01-10', 'Phone'),

    (2, DATE '2024-01-02', 'Tablet'),
    (2, DATE '2024-01-08', 'Tablet'),

    (3, DATE '2024-01-03', 'Desktop'),
    (3, DATE '2024-01-06', 'Desktop'),
    (3, DATE '2024-01-09', 'Desktop'),

    (4, DATE '2024-01-04', 'Mobile'),
    (4, DATE '2024-01-11', 'Laptop'),

    (5, DATE '2024-01-05', 'Watch'),
    (5, DATE '2024-01-07', 'Watch')
AS login_history(user_id, login_date, device);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW trusted_devices AS
SELECT *
FROM VALUES
    (1, 'Laptop'),
    (2, 'Tablet'),
    (3, 'Desktop'),
    (5, 'Watch')
AS trusted_devices(user_id, device);

In [0]:
%sql
select * from login_history

In [0]:
%sql
select * from trusted_devices

In [0]:
%sql
with untrusted_users (
select * from login_history l where not exists 
(
    select 1 from trusted_devices t where l.user_id = t.user_id and l.device = t.device
)
)
select distinct user_id, device from login_history l where not exists 
(
    select 1 from untrusted_users u where u.user_id = l.user_id
)